In [1]:
from dotenv import load_dotenv
import os
load_dotenv(r"D:\GPT_AGENT_2025_BOOK\chap02\.env")
api_key=os.getenv("OPENAI_API_KEY")

In [2]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

llm=ChatOpenAI(model="gpt-4o-mini")
llm.invoke([HumanMessage("잘 지냈어?")])

AIMessage(content='네, 고맙습니다! 당신은 어떻게 지내고 계신가요? 도움이 필요하시면 말씀해 주세요.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 12, 'total_tokens': 38, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_560af6e559', 'id': 'chatcmpl-CFXNdSMwdZBKOqA6QiJUGGSGzTRA7', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--53200586-5f1a-4d07-b170-db092f3dfbf8-0', usage_metadata={'input_tokens': 12, 'output_tokens': 26, 'total_tokens': 38, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [ ]:
from langchain_core.tools import tool
from datetime import datetime
import pytz

@tool   # @tool 데코레이터를 사용하여 함수를 도구로 등록
def get_current_time(timezone: str, location: str) -> str:
    """ 현재 시각을 반환하는 함수

    Args:
        timezone (str): 타임존(예: 'Asia/Seoul'). 실제 존재해야 함
        location (str): 지역명. 타임존은 모든 지명에 대응되지 않으므로 이후 llm 답변 생성에 사용됨
    """
    tz=pytz.timezone(timezone)
    now = datetime.now(tz).strftime("%Y-%m-%d %H:%M:%S")
    location_and_local_time = f'{timezone} ({location}) 현재시각 {now} ' # 타임존, 지역명, 현재시각을 문자열로 반환
    print(location_and_local_time)
    return location_and_local_time

In [4]:
# 도구를 tools 리스트에 추가하고, tool_dict에도 추가
tools=[get_current_time,]
tool_dict={"get_current_time":get_current_time,}

# 도구를 모델에 바인딩: 모델에 도구를 바인딩하면, 도구를 사용하여 llm 답변을 생성할 수 있음
llm_with_tools=llm.bind_tools(tools)

In [5]:
from langchain_core.messages import SystemMessage

messages=[
    SystemMessage("너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다."),
    HumanMessage("부산은 지금 몇 시야?")
]

response=llm_with_tools.invoke(messages)
messages.append(response)

print(messages)

[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다.', additional_kwargs={}, response_metadata={}), HumanMessage(content='부산은 지금 몇 시야?', additional_kwargs={}, response_metadata={}), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_41KYowSQ2vxGCsvbGa4rWzxb', 'function': {'arguments': '{"timezone":"Asia/Seoul","location":"Busan"}', 'name': 'get_current_time'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 130, 'total_tokens': 153, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_51db84afab', 'id': 'chatcmpl-CFXaritVKz4XFykWLlb7yg2HyMNwF', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--907164a6-f7e5-43cb-90bf-bf69a4950845-0', tool_c

In [6]:
for tool_call in response.tool_calls:
    selected_tool = tool_dict[tool_call["name"]]
    print(tool_call["args"])
    tool_msg=selected_tool.invoke(tool_call)
    messages.append(tool_msg)

messages

{'timezone': 'Asia/Seoul', 'location': 'Busan'}
Asia/Seoul (Busan) 현재시각 2025-09-14 12:28:53 


[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='부산은 지금 몇 시야?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_41KYowSQ2vxGCsvbGa4rWzxb', 'function': {'arguments': '{"timezone":"Asia/Seoul","location":"Busan"}', 'name': 'get_current_time'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 130, 'total_tokens': 153, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_51db84afab', 'id': 'chatcmpl-CFXaritVKz4XFykWLlb7yg2HyMNwF', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--907164a6-f7e5-43cb-90bf-bf69a4950845-0', tool

In [7]:
llm_with_tools.invoke(messages)

AIMessage(content='부산은 지금 2025년 9월 14일 12시 28분 53초입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 187, 'total_tokens': 213, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_51db84afab', 'id': 'chatcmpl-CFXjpYPe8MQdGkGeoygo5oco4zqmB', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--d819a96d-680d-4c66-ac5e-60126b72561e-0', usage_metadata={'input_tokens': 187, 'output_tokens': 26, 'total_tokens': 213, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [8]:
from pydantic import BaseModel, Field

class StockHistoryInput(BaseModel):
    ticker: str = Field(..., title="주식 코드", description="주식 코드 (예: AAPL)")
    period: str = Field(..., title="기간", description="주식 데이터 조회 기간 (예: 1d, 1mo, 1y)")

In [9]:
import yfinance as yf

@tool
def get_yf_stock_history(stock_history_input: StockHistoryInput) -> str:
    """ 주식 종목의 가격 데이터를 조회하는 함수"""
    stock = yf.Ticker(stock_history_input.ticker)
    history = stock.history(period=stock_history_input.period)
    history_md = history.to_markdown() 

    return history_md

tools = [get_current_time, get_yf_stock_history]
tool_dict = {"get_current_time": get_current_time, "get_yf_stock_history": get_yf_stock_history}

llm_with_tools = llm.bind_tools(tools)

In [10]:
messages.append(HumanMessage("테슬라는 한 달 전에 비해 주가가 올랐나 내렸나?"))

response = llm_with_tools.invoke(messages)
print(response)
messages.append(response)

content='' additional_kwargs={'tool_calls': [{'id': 'call_VpmhalrO0p1PQyzAXZKnuVUP', 'function': {'arguments': '{"stock_history_input":{"ticker":"TSLA","period":"1mo"}}', 'name': 'get_yf_stock_history'}, 'type': 'function'}], 'refusal': None} response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 278, 'total_tokens': 305, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_560af6e559', 'id': 'chatcmpl-CFXwAVmyAHs2SdsEF9rCK0HlTV19T', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None} id='run--a126f116-dd9f-4d12-bd8c-c71c2c558e7e-0' tool_calls=[{'name': 'get_yf_stock_history', 'args': {'stock_history_input': {'ticker': 'TSLA', 'period': '1mo'}}, 'id': 'call_VpmhalrO0p1PQyzAXZKnuVUP', 'type': 'tool_call'}] usage_metadata={'inp

In [11]:
for tool_call in response.tool_calls:
    selected_tool = tool_dict[tool_call["name"]]
    print(tool_call["args"])
    tool_msg = selected_tool.invoke(tool_call)
    messages.append(tool_msg)
    print(tool_msg)

{'stock_history_input': {'ticker': 'TSLA', 'period': '1mo'}}
content='| Date                      |   Open |   High |    Low |   Close |      Volume |   Dividends |   Stock Splits |\n|:--------------------------|-------:|-------:|-------:|--------:|------------:|------------:|---------------:|\n| 2025-08-13 00:00:00-04:00 | 341.5  | 348.98 | 338.2  |  339.38 | 6.78389e+07 |           0 |              0 |\n| 2025-08-14 00:00:00-04:00 | 335.76 | 340.47 | 330.4  |  335.58 | 7.50007e+07 |           0 |              0 |\n| 2025-08-15 00:00:00-04:00 | 337.66 | 339.3  | 327.02 |  330.56 | 7.43198e+07 |           0 |              0 |\n| 2025-08-18 00:00:00-04:00 | 329.62 | 336.27 | 329.59 |  335.16 | 5.69566e+07 |           0 |              0 |\n| 2025-08-19 00:00:00-04:00 | 335.79 | 340.55 | 327.85 |  329.31 | 7.5956e+07  |           0 |              0 |\n| 2025-08-20 00:00:00-04:00 | 329.22 | 331.37 | 314.6  |  323.9  | 7.74818e+07 |           0 |              0 |\n| 2025-08-21 00:00:00-04:0

In [12]:
llm_with_tools.invoke(messages)

AIMessage(content='테슬라의 주가는 한 달 전과 비교해 다음과 같습니다:\n\n- **한 달 전 주가 (2025년 8월 13일)**: $339.38\n- **현재 주가 (2025년 9월 12일)**: $395.94\n\n따라서, 테슬라는 한 달 전에 비해 주가가 상승했습니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 81, 'prompt_tokens': 1646, 'total_tokens': 1727, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_560af6e559', 'id': 'chatcmpl-CFY00tLXfYqVw6iM3kgosxmIWyKHo', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--6043e5c4-b98b-4f6e-b413-13cda2e16d4b-0', usage_metadata={'input_tokens': 1646, 'output_tokens': 81, 'total_tokens': 1727, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})